# Visualization of simulation

In [ ]:
%matplotlib notebook

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import pickle
import pandas as pd
from tqdm import tqdm

# --------------------------
# Load simulation data
# --------------------------
with open("simulation_data.pkl", "rb") as f:
    df = pickle.load(f)

# Separate bees and plants
bee_df = df[df["Agent Type"] == "bee"]
plant_df = df[(df.index.get_level_values("Step") == 0) & (df["Agent Type"] == "plant")]
steps = sorted(bee_df.index.get_level_values("Step").unique())

# --------------------------
# Plot setup
# --------------------------
fig, ax = plt.subplots(figsize=(72/4, 18/4))
ax.set_facecolor("gray")

# Plot environment borders (72 x 0.7 = 50.4, 18 x 0.7 = 12.6)
x_min, x_max = 0, 72 * 0.7
y_min, y_max = 0, 18 * 0.7
ax.plot([x_min, x_max], [y_min, y_min], color="black", linewidth=1.5)
ax.plot([x_min, x_max], [y_max, y_max], color="black", linewidth=1.5)
ax.plot([x_min, x_min], [y_min, y_max], color="black", linewidth=1.5)
ax.plot([x_max, x_max], [y_min, y_max], color="black", linewidth=1.5)

# Plot static plants using "Plot Code"
for _, plant in plant_df.iterrows():
    pos = plant["Bee Positions"]
    style = plant["Plot Code"]
    assert isinstance(style, dict), f"Invalid Plot Code in plant: {style}"
    ax.scatter(pos[0], pos[1],
               marker=style.get("marker", "o"),
               c=style.get("color"),
               edgecolors=style.get("edgecolors", "black"),
               linewidths=style.get("linewidths", 0.8),
               s=40, zorder=1)

# Set axis limits
all_bee_positions = bee_df["Bee Positions"].tolist()
plant_positions = plant_df["Bee Positions"].tolist()
all_x = [p[0] for p in all_bee_positions + plant_positions]
all_y = [p[1] for p in all_bee_positions + plant_positions]
ax.set_xlim(min(all_x) - 1, max(all_x) + 1)
ax.set_ylim(min(all_y) - 1, max(all_y) + 1)
ax.set_xlabel("X Position")
ax.set_ylabel("Y Position")
ax.set_title("Bee Movement Relative to Plants")

ax.set_xlim(x_min - 2, x_max + 2)
ax.set_ylim(y_min - 2, y_max + 2)

# --------------------------
# Animation logic
# --------------------------
bee_scatters = []

def init():
    global bee_scatters
    bee_scatters = []
    return []

def update(frame):
    global bee_scatters

    # Clear previous bee visuals
    for scatter in bee_scatters:
        scatter.remove()
    bee_scatters.clear()

    if frame not in bee_df.index.get_level_values("Step"):
        return []

    current_bees = bee_df.loc[frame]
    if isinstance(current_bees, pd.Series):
        current_bees = pd.DataFrame([current_bees])

    for _, bee in current_bees.iterrows():
        pos = bee["Bee Positions"]
        style = bee["Plot Code"]
        assert isinstance(style, dict), f"Invalid Plot Code in bee at step {frame}: {style}"
        scat = ax.scatter(pos[0], pos[1],
                          marker=style.get("marker", "o"),
                          c=style.get("color"),
                          edgecolors=style.get("edgecolors", "black"),
                          linewidths=style.get("linewidths", 0.8),
                          s=40, zorder=2)
        bee_scatters.append(scat)

    ax.set_title(f"Step {frame} - Bee Movement Relative to Plants")
    return bee_scatters

# --------------------------
# Create animation
# --------------------------
frame_gen = (i for i in tqdm(steps, desc="Rendering animation"))

anim = FuncAnimation(fig, update, frames=frame_gen, init_func=init,
                     blit=False, interval=300, repeat=False)

HTML(anim.to_jshtml())
